# Interp-Research Experiment Analysis

Analysis of autonomous attribution quality optimization results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (6 columns: commit, span_f1, probe_f1, mean_iou, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["span_f1"] = pd.to_numeric(df["span_f1"], errors="coerce")
df["probe_f1"] = pd.to_numeric(df["probe_f1"], errors="coerce")
df["mean_iou"] = pd.to_numeric(df["mean_iou"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(f"  #{i:3d}  span_f1={row['span_f1']:.6f}  probe_f1={row['probe_f1']:.6f}  iou={row['mean_iou']:.6f}  {row['description']}")

## Span F1 Over Time

Track how span_f1 evolves as experiments progress. The running maximum shows the frontier.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 12), sharex=True)

# Filter out crashes
valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)

baseline_f1 = valid.loc[0, "span_f1"]

# --- Top panel: Span F1 ---
ax = axes[0]

disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["span_f1"], c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["span_f1"], c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_f1 = valid.loc[kept_mask, "span_f1"]
running_max = kept_f1.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60", linewidth=2, alpha=0.7, zorder=3, label="Running best")

for idx, f1 in zip(kept_idx, kept_f1):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 40: desc = desc[:37] + "..."
    ax.annotate(desc, (idx, f1), textcoords="offset points", xytext=(6, 6), fontsize=7.5, color="#1a7a3a", alpha=0.9, rotation=25, ha="left", va="bottom")

ax.set_ylabel("Span F1 (higher is better)", fontsize=12)
ax.set_title(f"Attribution Quality Progress: {len(df)} Experiments, {len(kept_v)} Kept", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

# --- Bottom panel: Mean IoU ---
ax2 = axes[1]
ax2.scatter(disc.index, disc["mean_iou"], c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")
ax2.scatter(kept_v.index, kept_v["mean_iou"], c="#3498db", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

kept_iou = valid.loc[kept_mask, "mean_iou"]
running_max_iou = kept_iou.cummax()
ax2.step(kept_idx, running_max_iou, where="post", color="#2980b9", linewidth=2, alpha=0.7, zorder=3, label="Running best")

ax2.set_xlabel("Experiment #", fontsize=12)
ax2.set_ylabel("Mean IoU (higher is better)", fontsize=12)
ax2.legend(loc="lower right", fontsize=9)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("interp_progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to interp_progress.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline_f1 = df.iloc[0]["span_f1"]
best_f1 = kept["span_f1"].max()
best_row = kept.loc[kept["span_f1"].idxmax()]

print(f"Baseline span_f1:   {baseline_f1:.6f}")
print(f"Best span_f1:       {best_f1:.6f}")
print(f"Total improvement:  {best_f1 - baseline_f1:+.6f} ({(best_f1 - baseline_f1) / max(baseline_f1, 1e-6) * 100:+.2f}%)")
print(f"Best experiment:    {best_row['description']}")
print()
print(f"Best probe_f1:      {kept['probe_f1'].max():.6f}")
print(f"Best mean_iou:      {kept['mean_iou'].max():.6f}")
print()
print("Cumulative improvements:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: f1={row['span_f1']:.6f}  iou={row['mean_iou']:.6f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
kept["prev_f1"] = kept["span_f1"].shift(1)
kept["delta"] = kept["span_f1"] - kept["prev_f1"]

hits = kept.iloc[1:].copy()
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'F1':>10}  {'IoU':>10}  Description")
print("-" * 90)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['span_f1']:.6f}  {row['mean_iou']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  {'':>10}  TOTAL improvement over baseline")